In [73]:
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [74]:
df = pd.read_csv(
    'https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv'
)

df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [75]:
df.shape

(569, 33)

In [76]:
df = df.drop(columns=['id', 'Unnamed: 32'])

In [77]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [78]:
df['diagnosis'].value_counts()

diagnosis
B    357
M    212
Name: count, dtype: int64

In [79]:
X = df.drop(columns=['diagnosis'])

In [80]:
y = df['diagnosis']

In [81]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

In [82]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [83]:
encoder = LabelEncoder()

y_train = encoder.fit_transform(y_train)

y_test = encoder.transform(y_test)

In [84]:
y_test[:5]

array([1, 0, 0, 1, 0])

In [85]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

y_train_tensor = y_train_tensor.reshape(-1, 1)
y_test_tensor = y_test_tensor.reshape(-1, 1)

In [ ]:
## simple NN model

class MySimpleNN():
    
    def __init__(self, X):
        
        self.weights = torch.rand(X.shape[1], 1, requires_grad=True, dtype=torch.float32)
        self.bias = torch.zeros(1, requires_grad=True, dtype=torch.float32)
    
    def forward(self, X):
        return torch.sigmoid(torch.matmul(X, self.weights) + self.bias)
    
    def loss_function(self, y_pred, y):

        loss = torch.nn.functional.binary_cross_entropy(
            y_pred,
            y
        )

        return loss

        

In [92]:
config = {
    'learning_rate': 0.01,
    'epochs': 1000
}

In [88]:
## Training model

model = MySimpleNN(X_train_tensor)
print(model.weights.shape)

torch.Size([30, 1])


In [96]:
for epoch in range(config['epochs']):
    
    ## forward pass 
    y_pred = model.forward(X_train_tensor)
    
    ## loss
    loss = model.loss_function(y_pred, y_train_tensor)
    
    
    ## backwad pass
    loss.backward()
    
    ## update weights
    with torch.no_grad():
        model.weights.data -= config['learning_rate'] * model.weights.grad
        model.bias.data -= config['learning_rate'] * model.bias.grad
    
    ## zero grad
    
    model.weights.grad.zero_()
    model.bias.grad.zero_()
    
    ## print loss 
    if epoch % 100 == 0:
        print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")
    
    
            

Epoch: 1, Loss: 0.08284799754619598
Epoch: 101, Loss: 0.081151083111763
Epoch: 201, Loss: 0.07980065792798996
Epoch: 301, Loss: 0.07869024574756622
Epoch: 401, Loss: 0.07775075733661652
Epoch: 501, Loss: 0.07693622261285782
Epoch: 601, Loss: 0.07621537148952484
Epoch: 701, Loss: 0.07556652277708054
Epoch: 801, Loss: 0.07497435063123703
Epoch: 901, Loss: 0.07442784309387207


In [94]:
model.weights

tensor([[ 1.0451],
        [ 0.3623],
        [ 0.3543],
        [ 0.3308],
        [-0.0059],
        [-0.0540],
        [ 0.5076],
        [ 0.3580],
        [ 0.3293],
        [ 0.3312],
        [ 0.8434],
        [ 0.0427],
        [ 0.4167],
        [ 0.9131],
        [-0.2292],
        [-0.3600],
        [-0.0067],
        [-0.2611],
        [ 0.0043],
        [-0.1032],
        [ 0.8842],
        [ 0.9122],
        [ 0.7984],
        [ 0.8776],
        [ 0.4062],
        [ 0.3754],
        [ 0.3610],
        [ 0.3229],
        [ 0.3501],
        [ 0.4499]], requires_grad=True)

In [100]:
## prediction 

with torch.no_grad():
    
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.5).float()

    accuracy = torch.sum(y_pred == y_test_tensor).item() / len(y_test_tensor)
    
    print(f"Accuracy: { round(accuracy*100 , 2)}")

Accuracy: 96.49
